# DeepNet+ / TGN dataset build

Builds the model-ready dataset defined in *DeepNet+ / TGN Thesis: Dataset Specification* from
the raw CAIDA files. Everything is derived from **real ITDK nodes and links**; nothing is invented.

| Stage | What happens | Spec |
|---|---|---|
| 1 | Load Ark traceroutes, verify RTT unit, keep raw and processed RTT separate | 5 |
| 2 | Map hop IPs to ITDK node IDs (stream `.nodes`) | 15.3 |
| 3 | Map consecutive hops to real ITDK links (stream `.links`); log every failure | 15.6, 15.7 |
| 4 | Split clean paths into 15 graphs, mark observed / unobserved paths, fix the graph-level split | 7, 8 |
| 5 | Node features, edge features (never invented), tomography matrix `A`, TGN events | 4, 10, 12 |
| 6 | Statistics, normalisation from **train graphs only**, config, provenance, acceptance checks | 17-19 |

Design rules that follow from the spec:
- A path is used only if **every** hop maps to an ITDK node, hops are contiguous, and every consecutive
  pair is an ITDK link. No anonymous or synthetic nodes. Failures are saved with a reason.
- Repeated measurements are **kept**, one path row per traceroute.
- Hidden (unobserved) paths keep their true RTT only in `path_ground_truth.csv`, never in model inputs.
- Packet loss is `NA` (`unavailable`): the traceroute file has one probe per hop and no probe counts.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import json
import re

import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RTT_NAME = "caida_real_rtt_supervised.csv"
if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/My Drive/Thesis/DeepNT_Output")
    ITDK_DIR = Path("/content/drive/My Drive/DataSet")
    THESIS = Path("/content/drive/My Drive/Thesis")
else:                                              # running on this Windows machine
    THESIS = Path(r"C:\Users\KINGDOM\Desktop\Thesis")
    PROJECT_ROOT = THESIS / "DeepNT_Output"
    ITDK_DIR = THESIS / "DataSet"

NODES_FILE = ITDK_DIR / "midar-iff-snmp.nodes"
LINKS_FILE = ITDK_DIR / "midar-iff-snmp.links"
rtt_candidates = [PROJECT_ROOT / "real_rtt" / RTT_NAME, ITDK_DIR / RTT_NAME, THESIS / "RTT" / RTT_NAME]
RTT_FILE = next((c for c in rtt_candidates if c.exists()), rtt_candidates[0])
DATA_DIR = PROJECT_ROOT / "data"                 # folder layout of spec section 13
PROC = DATA_DIR / "processed"

NUM_GRAPHS = 15
SEED = 2026
OBSERVATION_RATE = 0.30                          # share of each graph's paths whose RTT the model may see
REQUIRE_ITDK_LINK = True                         # False keeps unverified edges, flagged (breaks spec rule 20)
RTT_MIN_MS, RTT_MAX_MS = 0.05, 5000.0            # documented QC rule: outside this range is flagged, not removed
MEASUREMENT_METHOD = "ark_team_probing_traceroute"
RTT_SOURCE = "caida_ark_warts:destination_rtt_end_to_end"

assert NUM_GRAPHS == 15, "the graph-level split below is defined for 15 graphs"
GRAPH_IDS = [f"graph_{i:02d}" for i in range(1, NUM_GRAPHS + 1)]
SPLIT = {g: "train" if i <= 10 else "val" if i <= 12 else "test" for i, g in enumerate(GRAPH_IDS, 1)}

print("running in", "Colab" if IN_COLAB else "local Windows")
missing = []
for label, p in [("RTT", RTT_FILE), ("nodes", NODES_FILE), ("links", LINKS_FILE)]:
    print(f"[{'OK     ' if p.exists() else 'MISSING'}] {label:6s} {p}")
    if not p.exists():
        missing.append(p)
if missing:
    raise FileNotFoundError(f"missing input files: {[str(m) for m in missing]}. "
                            f"Put {RTT_NAME} in one of {[str(c.parent) for c in rtt_candidates]} or run in Colab.")

for d in [PROC / "graphs", PROC / "tomography", PROC / "tgn", DATA_DIR / "raw",
          PROJECT_ROOT / "experiments" / "real_rtt", PROJECT_ROOT / "experiments" / "deepnet",
          PROJECT_ROOT / "experiments" / "tgn"]:
    d.mkdir(parents=True, exist_ok=True)


## Stage 1: Ark traceroutes and RTT

The raw value and the millisecond value are kept in separate columns, and the unit is verified from the data
(the column is named `*_ms` but Scamper records microseconds).

In [ ]:
REQUIRED = ['measurement_id', 'warts_file', 'source_ip', 'destination_ip', 'start_time', 'hop_numbers',
            'path_ips', 'destination_responded', 'destination_hop_number', 'destination_rtt_ms']
raw = pd.read_csv(RTT_FILE, usecols=REQUIRED)
n_total = len(raw)

raw['rtt_raw'] = pd.to_numeric(raw['destination_rtt_ms'], errors='coerce')
median_raw = raw['rtt_raw'].median()
RTT_RAW_UNIT = 'microseconds' if median_raw > 1000 else 'milliseconds'
raw['rtt_ms'] = raw['rtt_raw'] / (1000.0 if RTT_RAW_UNIT == 'microseconds' else 1.0)
print(f"{n_total:,} traceroutes | median raw value {median_raw:,.0f} -> unit '{RTT_RAW_UNIT}' | "
      f"median RTT {raw['rtt_ms'].median():.1f} ms")

raw['monitor_id'] = raw['warts_file'].astype(str).str.extract(r'([A-Za-z0-9]+-[a-z]{2})\.team-probing')[0]
raw['measurement_time'] = pd.to_datetime(raw['start_time'], errors='coerce', utc=True)
raw['responded'] = raw['destination_responded'].astype(str).str.lower().isin(['true', '1', '1.0', 'yes'])
raw['destination_hop_number'] = pd.to_numeric(raw['destination_hop_number'], errors='coerce')
print("monitors:", raw['monitor_id'].value_counts().to_dict())
print("time span:", raw['measurement_time'].min(), "->", raw['measurement_time'].max())

IPV4 = re.compile(r'\d{1,3}(?:\.\d{1,3}){3}')
INT = re.compile(r'\d+')


def ip_sequence(row):
    """([source, hop1 ... destination], None) only for a fully observed traceroute; else (None, reason)."""
    if not row.responded or pd.isna(row.rtt_ms):
        return None, 'no_destination_rtt'
    ips = IPV4.findall(str(row.path_ips))
    hops = [int(h) for h in INT.findall(str(row.hop_numbers))]
    if not ips or len(ips) != len(hops):
        return None, 'hop_parse_mismatch'
    if hops[0] != 1 or any(b != a + 1 for a, b in zip(hops, hops[1:])):
        return None, 'unresponsive_hop_gap'
    if ips[-1] != row.destination_ip:
        if row.destination_hop_number != len(hops) + 1:
            return None, 'destination_not_at_end'
        ips.append(row.destination_ip)
    return [row.source_ip] + ips, None


fail = {}          # row index -> (reason, detail): every rejected traceroute is recorded, none dropped silently
ip_seqs = {}
for row in raw.itertuples():
    seq, why = ip_sequence(row)
    if why:
        fail[row.Index] = (why, '')
    else:
        ip_seqs[row.Index] = seq
print(f"contiguous, destination answered: {len(ip_seqs):,} / {n_total:,}")


## Stage 2: hop IPs to ITDK node IDs

One streaming pass over `.nodes`, keeping only the IPs we need (memory stays small). Also records each node's
interface count for the node features.

In [ ]:
wanted_ips = {ip for seq in ip_seqs.values() for ip in seq}
print(f"resolving {len(wanted_ips):,} distinct IPs ...")

ip_to_node, node_ifaces = {}, {}
with open(NODES_FILE, 'r', encoding='utf-8', errors='replace') as f:
    for n_lines, line in enumerate(f, 1):
        if line[:4] == 'node':
            parts = line.split()
            node = parts[1].rstrip(':')
            hit = False
            for tok in parts[2:]:
                if tok in wanted_ips:
                    ip_to_node[tok] = node
                    hit = True
            if hit:
                node_ifaces[node] = len(parts) - 2
        if n_lines % 25_000_000 == 0:
            print(f"  ...{n_lines:,} lines, {len(ip_to_node):,} IPs resolved")
print(f"resolved {len(ip_to_node):,} / {len(wanted_ips):,} IPs to ITDK nodes")

node_paths, alias_collapsed = {}, {}
for idx, seq in ip_seqs.items():
    nodes, collapsed = [], False
    for hop, ip in enumerate(seq):
        node = ip_to_node.get(ip)
        if node is None:
            fail[idx] = ('ip_not_in_itdk', f"hop {hop} of {len(seq) - 1}")
            break
        if nodes and nodes[-1] == node:          # two interfaces of one router
            collapsed = True
            continue
        nodes.append(node)
    else:
        if len(nodes) < 2:
            fail[idx] = ('source_equals_destination', '')
        elif len(set(nodes)) != len(nodes):
            fail[idx] = ('loop_in_path', '')
        else:
            node_paths[idx] = nodes
            alias_collapsed[idx] = collapsed
print(f"every hop mapped to an ITDK node, simple path: {len(node_paths):,} / {len(ip_seqs):,}")


## Stage 3: consecutive hops to real ITDK links

Streams `.links` once. A hop pair is a real edge only if both routers appear on the same ITDK link; the ITDK
link ID is stored as `itdk_link_id`. This is the slow step (about 174 million links).

In [ ]:
def canon(a, b):
    return (a, b) if int(a[1:]) <= int(b[1:]) else (b, a)


need_adj = defaultdict(set)
for nodes in node_paths.values():
    for a, b in zip(nodes, nodes[1:]):
        need_adj[a].add(b)
        need_adj[b].add(a)
print(f"looking for {sum(len(v) for v in need_adj.values()) // 2:,} distinct hop pairs among {len(need_adj):,} nodes")

edge_links = defaultdict(list)        # (a, b) canonical -> ITDK link IDs containing both routers
with open(LINKS_FILE, 'r', encoding='utf-8', errors='replace') as f:
    for n_lines, line in enumerate(f, 1):
        if line[:4] == 'link':
            parts = line.split()
            hit = {tok.partition(':')[0] for tok in parts[2:]} & need_adj.keys()
            if len(hit) >= 2:
                link_id = parts[1].rstrip(':')
                for a in hit:
                    for b in need_adj[a] & hit:
                        key = canon(a, b)
                        if a == key[0] and len(edge_links[key]) < 5:
                            edge_links[key].append(link_id)
        if n_lines % 25_000_000 == 0:
            print(f"  ...{n_lines:,} lines, {len(edge_links):,} pairs confirmed")
print(f"confirmed as ITDK links: {len(edge_links):,} pairs")

clean = []
for idx, nodes in node_paths.items():
    edges = [canon(a, b) for a, b in zip(nodes, nodes[1:])]
    missing = [e for e in edges if e not in edge_links]
    if missing and REQUIRE_ITDK_LINK:
        fail[idx] = ('edge_not_in_itdk_links', f"{len(missing)} of {len(edges)} hop pairs")
        continue
    tags = (['alias_collapsed'] if alias_collapsed[idx] else []) + (['unverified_edge'] if missing else [])
    clean.append({'idx': idx, 'nodes': nodes, 'edges': edges, 'flag': ';'.join(tags) or 'ok'})

funnel = [('traceroutes in RTT file', n_total)]
running = n_total
for reason in ['no_destination_rtt', 'hop_parse_mismatch', 'unresponsive_hop_gap', 'destination_not_at_end',
               'ip_not_in_itdk', 'source_equals_destination', 'loop_in_path', 'edge_not_in_itdk_links']:
    n = sum(1 for r, _ in fail.values() if r == reason)
    running -= n
    funnel.append((f"  - rejected: {reason}", -n))
funnel.append(('usable paths (all ITDK nodes + real ITDK links)', len(clean)))
funnel_df = pd.DataFrame(funnel, columns=['stage', 'traceroutes'])
print(funnel_df.to_string(index=False))
print(f"\nusable: {len(clean):,} / {n_total:,} = {100 * len(clean) / n_total:.1f}%")
assert running == len(clean), "funnel does not add up"
if len(clean) < NUM_GRAPHS * 20:
    print("WARNING: fewer than ~20 paths per graph; see the funnel above for the dominant loss")

PROC.mkdir(parents=True, exist_ok=True)
funnel_df.to_csv(PROC / "mapping_funnel.csv", index=False)
(pd.DataFrame([(raw.at[i, 'measurement_id'], raw.at[i, 'source_ip'], raw.at[i, 'destination_ip'], r, d)
               for i, (r, d) in sorted(fail.items())],
              columns=['raw_measurement_id', 'source_ip', 'destination_ip', 'reason', 'detail'])
   .to_csv(PROC / "mapping_failures.csv", index=False))


## Stage 4: 15 graphs, observed / unobserved masks, graph-level split

Paths are sorted by their node sequence and cut into 15 consecutive chunks. Paths that share a route prefix
therefore land in the same graph, which keeps graphs coherent (edges are covered by several paths) and keeps
different graphs as separate parts of the routing tree. Graphs 1-10 train, 11-12 validation, 13-15 test.

In [ ]:
clean.sort(key=lambda p: tuple(int(n[1:]) for n in p['nodes']))
if len(clean) < NUM_GRAPHS:
    raise RuntimeError(f"only {len(clean)} usable paths, cannot build {NUM_GRAPHS} graphs; see mapping_funnel.csv")

graphs, path_rows = {}, []
for gno, chunk in enumerate(np.array_split(np.arange(len(clean)), NUM_GRAPHS), 1):
    gid = GRAPH_IDS[gno - 1]
    items = [clean[i] for i in chunk]
    G = nx.Graph()
    for it in items:
        for a, b in it['edges']:
            if not G.has_edge(a, b):
                G.add_edge(a, b, edge_id=f"{a}-{b}", link_ids=edge_links.get((a, b), []), path_count=0)
            G[a][b]['path_count'] += 1
    graphs[gid] = G

    n_obs = max(1, int(round(OBSERVATION_RATE * len(items))))
    observed = np.zeros(len(items), dtype=int)
    observed[np.random.default_rng(SEED + gno).choice(len(items), n_obs, replace=False)] = 1

    for k, it in enumerate(items):
        r = raw.loc[it['idx']]
        outlier = [] if RTT_MIN_MS <= r.rtt_ms <= RTT_MAX_MS else ['rtt_outlier_high' if r.rtt_ms > RTT_MAX_MS else 'rtt_outlier_low']
        path_rows.append({
            'graph_id': gid, 'path_id': f"{gid}_p{k:05d}", 'split': SPLIT[gid],
            'source_node': it['nodes'][0], 'destination_node': it['nodes'][-1],
            'path_nodes': '|'.join(it['nodes']),
            'path_edges': '|'.join(f"{a}-{b}" for a, b in it['edges']),
            'path_length': len(it['edges']),
            'measurement_time': r.measurement_time.isoformat() if pd.notna(r.measurement_time) else '',
            'monitor_id': r.monitor_id, 'measurement_method': MEASUREMENT_METHOD,
            'observed': int(observed[k]), 'raw_measurement_id': r.measurement_id,
            'rtt_source': RTT_SOURCE, 'structural_flag': it['flag'],
            'quality_flag': ';'.join([f for f in it['flag'].split(';') if f != 'ok'] + outlier) or 'ok',
            'rtt_ms': r.rtt_ms, 'rtt_raw': r.rtt_raw, 'rtt_raw_unit': RTT_RAW_UNIT,
        })

full = pd.DataFrame(path_rows)
print(full.groupby('graph_id').agg(paths=('path_id', 'size'), observed=('observed', 'sum'),
                                   nodes=('graph_id', lambda s: graphs[s.iloc[0]].number_of_nodes()),
                                   edges=('graph_id', lambda s: graphs[s.iloc[0]].number_of_edges())).to_string())


## Stage 5: features, path tables, tomography matrix, TGN events

Model inputs never contain a hidden RTT: `paths_all` and `paths_unobserved` carry no target columns, edge
features use only observed measurements, and true values of hidden paths live only in `path_ground_truth.csv`.
Edge capacity, utilisation, delay and loss are `NA` because nothing measures them.

In [10]:
# ---- node features (per graph, structure only) ----
node_rows = []
for gid, G in graphs.items():
    n = G.number_of_nodes()
    deg, clus, pr, core = dict(G.degree()), nx.clustering(G), nx.pagerank(G), nx.core_number(G)
    clo = nx.closeness_centrality(G) if n <= 5000 else {}
    bet = nx.betweenness_centrality(G, k=min(n, 300), seed=SEED)
    for node in G:
        node_rows.append({
            'graph_id': gid, 'node_id': node, 'interface_count': node_ifaces.get(node, np.nan),
            'degree': deg[node], 'norm_degree': deg[node] / max(n - 1, 1), 'clustering_coeff': clus[node],
            'pagerank': pr[node], 'closeness_centrality': clo.get(node, np.nan),
            'betweenness_centrality': bet[node], 'k_core': core[node],
            'asn': pd.NA, 'country': pd.NA, 'latitude': np.nan, 'longitude': np.nan})
node_features = pd.DataFrame(node_rows)

# ---- edge features (nothing invented) ----
obs_count = Counter()
for r in full[full['observed'] == 1].itertuples():
    for e in r.path_edges.split('|'):
        obs_count[(r.graph_id, e)] += 1
edge_rows = []
for gid, G in graphs.items():
    for _, _, d in sorted(G.edges(data=True), key=lambda x: x[2]['edge_id']):
        src, dst = d['edge_id'].split('-')       # canonical order; NetworkX's own (u, v) order is arbitrary
        edge_rows.append({
            'graph_id': gid, 'edge_id': d['edge_id'], 'src': src, 'dst': dst,
            'itdk_link_id': d['link_ids'][0] if d['link_ids'] else '',
            'itdk_link_ids': ';'.join(d['link_ids']),
            'capacity': np.nan, 'utilization': np.nan, 'base_delay': np.nan, 'loss_rate': np.nan,
            'path_count_all': d['path_count'],                       # structural, known for hidden paths too
            'observed_measurement_count': obs_count[(gid, d['edge_id'])]})
edge_features = pd.DataFrame(edge_rows)

# ---- path tables ----
STRUCT = ['graph_id', 'path_id', 'split', 'source_node', 'destination_node', 'path_nodes', 'path_edges',
          'path_length', 'measurement_time', 'monitor_id', 'measurement_method', 'observed',
          'raw_measurement_id']
paths_all = full[STRUCT + ['rtt_source']].assign(quality_flag=full['structural_flag'])
target_cols = ['rtt_ms', 'packet_loss', 'loss_flag', 'rtt_raw', 'rtt_raw_unit']
ground_truth = full[['path_id', 'graph_id', 'split', 'observed', 'rtt_ms', 'rtt_raw', 'rtt_raw_unit',
                     'rtt_source', 'quality_flag']].assign(packet_loss=np.nan, loss_flag='unavailable_no_probe_counts')
paths_observed = paths_all[paths_all['observed'] == 1].merge(
    ground_truth[['path_id', 'rtt_ms', 'rtt_raw', 'rtt_raw_unit', 'packet_loss', 'loss_flag']], on='path_id')
paths_observed['quality_flag'] = ground_truth.set_index('path_id').loc[paths_observed['path_id'], 'quality_flag'].to_numpy()
paths_unobserved = paths_all[paths_all['observed'] == 0]
assert not set(target_cols) & set(paths_unobserved.columns) and not set(target_cols) & set(paths_all.columns)

# ---- tomography A per graph (rows = paths in paths_all order, cols = edges in edge_features order) ----
A_blocks, row_index, col_index = [], [], []
for gid in GRAPH_IDS:
    prow = paths_all[paths_all['graph_id'] == gid].reset_index(drop=True)
    ecol = edge_features[edge_features['graph_id'] == gid].reset_index(drop=True)
    col_of = {e: i for i, e in enumerate(ecol['edge_id'])}
    r_i, c_i = [], []
    for i, pe in enumerate(prow['path_edges']):
        for e in pe.split('|'):
            r_i.append(i)
            c_i.append(col_of[e])
    A = sp.csr_matrix((np.ones(len(r_i), dtype=np.int8), (r_i, c_i)), shape=(len(prow), len(ecol)))
    assert (np.asarray(A.sum(axis=1)).ravel() == prow['path_length'].to_numpy()).all()
    sp.save_npz(PROC / "tomography" / f"A_{gid}.npz", A)
    A_blocks.append(A)
    row_index += [(gid, p, i) for i, p in enumerate(prow['path_id'])]
    col_index += [(gid, e, i) for i, e in enumerate(ecol['edge_id'])]
sp.save_npz(PROC / "tomography" / "A_matrix.npz", sp.block_diag(A_blocks, format='csr'))
pd.DataFrame(row_index, columns=['graph_id', 'path_id', 'row']).to_csv(PROC / "tomography" / "A_rows.csv", index=False)
pd.DataFrame(col_index, columns=['graph_id', 'edge_id', 'col']).to_csv(PROC / "tomography" / "A_cols.csv", index=False)

# ---- TGN events: only real, timestamped, observed measurements ----
ev = paths_observed[paths_observed['measurement_time'] != ''].sort_values('measurement_time')
events = pd.DataFrame({
    'event_id': [f"ev_{i:06d}" for i in range(len(ev))], 'timestamp': ev['measurement_time'].to_numpy(),
    'source_node': ev['source_node'].to_numpy(), 'destination_node': ev['destination_node'].to_numpy(),
    'event_type': 'path_rtt_measurement', 'value_rtt_ms': ev['rtt_ms'].to_numpy(),
    'provenance': RTT_SOURCE, 'quality_flag': ev['quality_flag'].to_numpy(),
    'graph_id': ev['graph_id'].to_numpy(), 'path_id': ev['path_id'].to_numpy()})
print(f"path table {len(paths_all):,} | observed {len(paths_observed):,} | hidden {len(paths_unobserved):,} | "
      f"edges {len(edge_features):,} | events {len(events):,}")


path table 1,326 | observed 396 | hidden 930 | edges 6,598 | events 396


## Stage 6: statistics, normalisation, provenance, save, acceptance checks

In [11]:
# ---- graph statistics incl. overlap between graphs (needed to interpret the inductive split) ----
node_sets = {g: set(G) for g, G in graphs.items()}
edge_sets = {g: {d['edge_id'] for _, _, d in G.edges(data=True)} for g, G in graphs.items()}
jac = lambda a, b: len(a & b) / max(len(a | b), 1)
stat_rows = []
for gid, G in graphs.items():
    comps = list(nx.connected_components(G))
    degs = [d for _, d in G.degree()]
    p = paths_all[paths_all['graph_id'] == gid]
    others = [g for g in GRAPH_IDS if g != gid]
    stat_rows.append({
        'graph_id': gid, 'split': SPLIT[gid], 'nodes': G.number_of_nodes(), 'edges': G.number_of_edges(),
        'density': nx.density(G), 'components': len(comps), 'largest_component': max(map(len, comps)),
        'mean_degree': float(np.mean(degs)), 'max_degree': max(degs), 'paths': len(p),
        'observed_paths': int(p['observed'].sum()), 'unobserved_paths': int((p['observed'] == 0).sum()),
        'monitors': ';'.join(sorted(p['monitor_id'].dropna().unique())),
        'max_node_jaccard_with_other_graph': max(jac(node_sets[gid], node_sets[o]) for o in others),
        'max_edge_jaccard_with_other_graph': max(jac(edge_sets[gid], edge_sets[o]) for o in others)})
graph_statistics = pd.DataFrame(stat_rows)
print(graph_statistics.drop(columns=['monitors']).round(3).to_string(index=False))

# ---- normalisation from TRAIN graphs only ----
feat_cols = ['interface_count', 'degree', 'norm_degree', 'clustering_coeff', 'pagerank',
             'closeness_centrality', 'betweenness_centrality', 'k_core']
train_nodes = node_features[node_features['graph_id'].map(SPLIT) == 'train']
train_y = np.log1p(paths_observed.loc[(paths_observed['split'] == 'train') &
                                      (paths_observed['quality_flag'].str.contains('rtt_outlier') == False), 'rtt_ms'])
normalization = {
    'fit_on': 'train graphs only',
    'node_features': {c: {'mean': float(train_nodes[c].mean()), 'std': float(train_nodes[c].std() or 1.0)} for c in feat_cols},
    'log1p_rtt_ms': {'mean': float(train_y.mean()), 'std': float(train_y.std())}}


# ---- provenance ----
def itdk_provenance(links_file):
    dates, start = [], None
    with open(links_file, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if not line.startswith('#'):
                break
            if 'start time' in line and start is None:
                start = line.strip()
            m = re.search(r'\.team-probing\.c\d+\.(\d{8})\.warts', line)
            if m:
                dates.append(m.group(1))
    return {'links_header': start, 'input_warts_files': len(dates),
            'input_first_date': min(dates) if dates else None, 'input_last_date': max(dates) if dates else None}


prov = itdk_provenance(LINKS_FILE)
rtt_days = sorted(raw['measurement_time'].dropna().dt.strftime('%Y%m%d').unique())
disjoint = bool(prov['input_last_date'] and rtt_days and rtt_days[0] > prov['input_last_date'])
config = {
    'created': pd.Timestamp.now(tz='UTC').isoformat(), 'seed': SEED, 'num_graphs': NUM_GRAPHS,
    'observation_rate': OBSERVATION_RATE, 'require_itdk_link': REQUIRE_ITDK_LINK,
    'rtt_qc_range_ms': [RTT_MIN_MS, RTT_MAX_MS], 'rtt_raw_unit': RTT_RAW_UNIT,
    'graph_split': SPLIT, 'path_length_convention': 'number of edges (hops)',
    'target_y': 'end-to-end round-trip time to the destination, in ms (not one-way, not additive per edge)',
    'path_selection_rule': 'destination answered; hops 1..k contiguous; every hop an ITDK node; simple path; every hop pair an ITDK link',
    'graph_sampling_rule': 'usable paths sorted by node sequence and cut into 15 equal consecutive chunks',
    'packet_loss': 'unavailable: traceroute has one probe per hop and no probe counts',
    'source_files': {'rtt_csv': str(RTT_FILE), 'rtt_csv_sha256': hashlib.sha256(RTT_FILE.read_bytes()).hexdigest(),
                     'nodes': str(NODES_FILE), 'links': str(LINKS_FILE)},
    'itdk_provenance': prov, 'rtt_measurement_days': rtt_days,
    'rtt_measured_after_itdk_input_window': disjoint,
    'funnel': {a.strip(): int(b) for a, b in funnel}}

# ---- save ----
for gid, G in graphs.items():
    (edge_features[edge_features['graph_id'] == gid][['src', 'dst', 'edge_id', 'itdk_link_id']]
        .rename(columns={'src': 'source', 'dst': 'target'}).to_csv(PROC / "graphs" / f"{gid}.csv", index=False))
node_features.to_csv(PROC / "node_features.csv", index=False)
edge_features.to_csv(PROC / "edge_features.csv", index=False)
paths_all.to_csv(PROC / "paths_all.csv", index=False)
paths_observed.to_csv(PROC / "paths_observed.csv", index=False)
paths_unobserved.to_csv(PROC / "paths_unobserved.csv", index=False)
ground_truth.to_csv(PROC / "path_ground_truth.csv", index=False)
graph_statistics.to_csv(PROC / "graph_statistics.csv", index=False)
events.to_csv(PROC / "tgn" / "events.csv", index=False)
(PROC / "normalization.json").write_text(json.dumps(normalization, indent=2))
(DATA_DIR / "config.json").write_text(json.dumps(config, indent=2, default=str))
print("saved to", DATA_DIR)


graph_id split  nodes  edges  density  components  largest_component  mean_degree  max_degree  paths  observed_paths  unobserved_paths  max_node_jaccard_with_other_graph  max_edge_jaccard_with_other_graph
graph_01 train    414    419    0.005           1                414        2.024          30     89              27                62                              0.051                              0.045
graph_02 train    339    346    0.006           1                339        2.041          23     89              27                62                              0.026                              0.021
graph_03 train    405    411    0.005           1                405        2.030          22     89              27                62                              0.051                              0.045
graph_04 train    498    500    0.004           1                498        2.008          19     89              27                62                              0.024           

In [12]:
# ---- acceptance criteria (spec section 19) ----
gt = ground_truth.set_index('path_id')
checks = {
    '15 graph files exist': all((PROC / "graphs" / f"{g}.csv").exists() for g in GRAPH_IDS),
    'every edge_id equals src-dst (canonical: lower ITDK node number first)': bool(
        (edge_features['edge_id'] == edge_features['src'] + '-' + edge_features['dst']).all()
        and (edge_features['src'].str[1:].astype(int) < edge_features['dst'].str[1:].astype(int)).all()),
    'every edge carries an ITDK link ID': bool((edge_features['itdk_link_id'] != '').all()) if REQUIRE_ITDK_LINK else True,
    'every observed path traces to a raw measurement': bool(
        paths_observed['raw_measurement_id'].isin(raw['measurement_id']).all()),
    'RTT unit verified and stored in ms': RTT_RAW_UNIT in ('microseconds', 'milliseconds') and bool(gt['rtt_ms'].between(0, 1e6).all()),
    'packet loss explicitly unavailable, not filled': bool(ground_truth['packet_loss'].isna().all()),
    'observed/unobserved masks explicit and disjoint': set(paths_all['observed']) <= {0, 1}
        and not set(paths_observed['path_id']) & set(paths_unobserved['path_id']),
    'graph-level split fixed, no graph in two splits': paths_all.groupby('graph_id')['split'].nunique().eq(1).all(),
    'A_matrix.npz rebuilt from path_edges: shape and edge count match': (lambda M: M.shape == (len(paths_all), len(edge_features))
        and int(M.sum()) == int(paths_all['path_length'].sum()))(sp.load_npz(PROC / 'tomography' / 'A_matrix.npz')),
    'no target column in model-input tables': not {'rtt_ms', 'rtt_raw', 'packet_loss'} & (
        set(paths_all.columns) | set(paths_unobserved.columns) | set(edge_features.columns) | set(node_features.columns)),
    'edge features use only observed measurements': int(edge_features['observed_measurement_count'].sum())
        == int(paths_observed['path_length'].sum()),
    'normalisation fitted on train graphs only': normalization['fit_on'] == 'train graphs only',
    'TGN events are real observed measurements with timestamps': bool(
        (events['timestamp'] != '').all() and events['path_id'].isin(paths_observed['path_id']).all()),
}
for name, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
print(f"\nRTT measured after the ITDK input window: {disjoint}  "
      f"(ITDK inputs {prov['input_first_date']}..{prov['input_last_date']}, RTT days {rtt_days})")
print("quality flags:", ground_truth['quality_flag'].value_counts().to_dict())
assert all(checks.values()), "an acceptance check failed"


[PASS] 15 graph files exist
[PASS] every edge_id equals src-dst (canonical: lower ITDK node number first)
[PASS] every edge carries an ITDK link ID
[PASS] every observed path traces to a raw measurement
[PASS] RTT unit verified and stored in ms
[PASS] packet loss explicitly unavailable, not filled
[PASS] observed/unobserved masks explicit and disjoint
[PASS] graph-level split fixed, no graph in two splits
[PASS] A_matrix.npz rebuilt from path_edges: shape and edge count match
[PASS] no target column in model-input tables
[PASS] edge features use only observed measurements
[PASS] normalisation fitted on train graphs only
[PASS] TGN events are real observed measurements with timestamps

RTT measured after the ITDK input window: True  (ITDK inputs 20250223..20250309, RTT days ['20250310', '20250311'])
quality flags: {'ok': 1307, 'alias_collapsed': 19}
